# LeetCode 2008 — Maximum Earnings From Taxi bằng Python

Link đề: https://leetcode.com/problems/maximum-earnings-from-taxi/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Dynamic Programming, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxTaxiEarnings(self, n: int, rides: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận diện bài **lập lịch khoảng có trọng số** (weighted interval scheduling): chọn các khoảng **không chồng nhau** để tổng trọng số lớn nhất.
2. Viết DP **theo điểm trên đường**: `dp[p]` = tiền lớn nhất khi đã tới điểm `p`, với "chuyển trạng thái" `dp[p] = max(dp[p−1], dp[start] + lợi nhuận)` cho các cuốc kết thúc tại `p`.
3. Viết DP **theo cuốc xe đã sắp xếp** + **tìm kiếm nhị phân** (`bisect`) — cách tổng quát khi tọa độ rất lớn.
4. Hiểu điểm tinh tế "được trả khách và đón khách khác **tại cùng một điểm**" ⇒ điều kiện không chồng là `end ≤ start` (không phải `<`).
5. Thấy vì sao các chiến lược **tham lam** quen thuộc thất bại, và kiểm chứng bằng brute force trên dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

Trên một con đường có các điểm `1..n`, taxi chạy từ điểm `1` tới điểm `n`, **không được quay đầu**. Mỗi hành khách `rides[i] = [start_i, end_i, tip_i]` muốn đi từ `start_i` tới `end_i` và boa `tip_i`. Nếu chở khách `i` bạn được `end_i − start_i + tip_i` đô. Mỗi lúc chỉ chở **tối đa một** khách. Bạn **được** trả khách và đón khách mới **tại cùng một điểm**.

Trả về số tiền lớn nhất có thể kiếm được.

**Ví dụ 1:** `n = 5, rides = [[2,5,4],[1,5,1]]` → `7` (chở khách 0: `5 − 2 + 4 = 7`).

**Ví dụ 2:** `n = 20, rides = [[1,6,1],[3,10,2],[10,12,3],[11,12,2],[12,15,2],[13,18,1]]` → `20`: chở khách 1 (`3→10`, 9 đô), khách 2 (`10→12`, 5 đô), khách 5 (`13→18`, 6 đô).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ len(rides) ≤ 3·10⁴`, `1 ≤ start_i < end_i ≤ n`, `1 ≤ tip_i ≤ 10⁵`.

In [1]:
import random
import time
from bisect import bisect_right, bisect_left
from collections import defaultdict
from itertools import combinations
from typing import List

examples = [
    (5, [[2, 5, 4], [1, 5, 1]], 7),
    (20, [[1, 6, 1], [3, 10, 2], [10, 12, 3], [11, 12, 2], [12, 15, 2], [13, 18, 1]], 20),
]
for n, rides, exp in examples:
    print(f"n={n}, rides={rides} -> kỳ vọng {exp}")

n=5, rides=[[2, 5, 4], [1, 5, 1]] -> kỳ vọng 7
n=20, rides=[[1, 6, 1], [3, 10, 2], [10, 12, 3], [11, 12, 2], [12, 15, 2], [13, 18, 1]] -> kỳ vọng 20


## 3. Kiến thức nền

### 3.1 Lợi nhuận một cuốc và điều kiện "không chồng"

Lợi nhuận `profit = end − start + tip`. Hai cuốc `a` rồi `b` (theo thứ tự đi) **tương thích** nếu `a.end ≤ b.start` — được phép bằng vì trả khách và đón khách tại cùng điểm.

In [2]:
def profit(r):
    s, e, t = r
    return e - s + t

def compatible(a, b):
    """a đi trước b?"""
    return a[1] <= b[0]

print([profit(r) for r in examples[1][1]])
print("(3,10) rồi (10,12) tương thích?", compatible([3, 10, 2], [10, 12, 3]))
print("(1,6) rồi (3,10) tương thích?", compatible([1, 6, 1], [3, 10, 2]))

[6, 9, 5, 3, 5, 6]
(3,10) rồi (10,12) tương thích? True
(1,6) rồi (3,10) tương thích? False


### 3.2 Nhóm theo khóa bằng `defaultdict(list)`

Để DP theo điểm, ta cần "các cuốc kết thúc tại điểm `p`": `by_end[e].append((s, profit))`.

In [3]:
by_end = defaultdict(list)
for s, e, t in examples[1][1]:
    by_end[e].append((s, e - s + t))
print(dict(by_end))

{6: [(1, 6)], 10: [(3, 9)], 12: [(10, 5), (11, 3)], 15: [(12, 5)], 18: [(13, 6)]}


### 3.3 `bisect_right` — đếm số phần tử `≤ x` trong mảng đã sắp

`bisect_right(a, x)` trả về vị trí chèn `x` vào **bên phải** mọi phần tử bằng `x` ⇒ đúng bằng **số phần tử `≤ x`**. `bisect_left` thì đếm số phần tử `< x`.

In [4]:
ends = [6, 10, 12, 12, 15, 18]
for x in [10, 11, 12, 3]:
    print(f"x={x}: số end ≤ x là {bisect_right(ends, x)}, số end < x là {bisect_left(ends, x)}")

x=10: số end ≤ x là 2, số end < x là 1
x=11: số end ≤ x là 2, số end < x là 2
x=12: số end ≤ x là 4, số end < x là 2
x=3: số end ≤ x là 0, số end < x là 0


### 3.4 Ý tưởng DP "chọn hoặc không chọn"

Với mỗi phần tử: **không chọn** ⇒ giữ đáp án trước đó; **chọn** ⇒ cộng giá trị của nó với đáp án tốt nhất của phần "không xung đột". DP lấy max hai lựa chọn — giống bài cái túi 0/1 nhưng "không xung đột" thay cho "còn đủ sức chứa".

In [5]:
# Ví dụ nhỏ: chọn các số không kề nhau để tổng lớn nhất (House Robber) — cùng khung "chọn/không chọn"
a = [2, 7, 9, 3, 1]
dp = [0] * (len(a) + 1)
for i in range(1, len(a) + 1):
    dp[i] = max(dp[i - 1], a[i - 1] + (dp[i - 2] if i >= 2 else 0))
print("House Robber:", dp[-1], dp)

House Robber: 12 [0, 2, 7, 11, 11, 12]


## 4. Ý tưởng nền tảng / trực giác

Hãy đi dọc con đường từ điểm 1 tới `n` và ghi lại `dp[p]` = số tiền **lớn nhất** có thể có khi xe ở điểm `p` (và đang rảnh).

- Đi tiếp một điểm mà không chở ai: `dp[p] ≥ dp[p−1]`.
- Nếu có cuốc `(s, p, tip)` kết thúc tại `p`: ta có thể đã rảnh ở `s` (tiền `dp[s]`) rồi chở khách tới `p`: `dp[p] ≥ dp[s] + (p − s + tip)`.

Tính tay ví dụ 2 (chỉ ghi các điểm thay đổi):
- `p = 6`: cuốc `(1,6)` ⇒ `dp[1] + 6 = 6`. `dp[6] = 6`.
- `p = 10`: cuốc `(3,10)` ⇒ `dp[3] + 9 = 0 + 9 = 9`. `dp[10] = max(6, 9) = 9`.
- `p = 12`: cuốc `(10,12)` ⇒ `9 + 5 = 14`; cuốc `(11,12)` ⇒ `dp[11] + 3 = 9 + 3 = 12`. `dp[12] = 14`.
- `p = 15`: `(12,15)` ⇒ `14 + 5 = 19`. `dp[15] = 19`.
- `p = 18`: `(13,18)` ⇒ `dp[13] + 6 = 14 + 6 = 20`. `dp[18] = 20`. Kéo tới `dp[20] = 20` ✓.

## 5. Cách 1 — Brute Force / baseline: thử mọi tập con cuốc xe

**Ý tưởng:** với mỗi tập con, sắp theo `start`, kiểm tra từng cặp liên tiếp tương thích, cộng lợi nhuận. Lấy max.

In [6]:
def earn_brute(n, rides):
    m = len(rides)
    best = 0
    for mask in range(1 << m):
        chosen = sorted((rides[i] for i in range(m) if mask >> i & 1), key=lambda r: r[0])
        if all(chosen[k][1] <= chosen[k + 1][0] for k in range(len(chosen) - 1)):
            best = max(best, sum(profit(r) for r in chosen))
    return best

for n, rides, exp in examples:
    got = earn_brute(n, rides); print(got, "kỳ vọng", exp); assert got == exp

7 kỳ vọng 7
20 kỳ vọng 20


**Độ phức tạp:** `O(2^m · m log m)` với `m = len(rides)`. `m = 3·10⁴` ⇒ không thể; chỉ dùng cho `m ≤ ~14`.

**Nút thắt:** ta xét lại cùng một "tình huống" (đang rảnh tại điểm `p`) rất nhiều lần qua các tập con khác nhau. Mà đáp án từ điểm `p` trở đi **không phụ thuộc** quá khứ ⇒ dùng DP.

## 6. Ý tưởng thuật toán / bước đột phá

**Cấu trúc con tối ưu:** gọi `dp[p]` = tiền tối đa từ các cuốc **kết thúc ≤ p**. Xét lời giải tối ưu cho `dp[p]`:
- **Nếu** nó không có cuốc nào kết thúc đúng tại `p` **thì** nó cũng là lời giải cho `dp[p−1]`.
- **Nếu** nó có cuốc `(s, p, tip)` **thì** các cuốc còn lại đều kết thúc `≤ s` ⇒ phần đó tối đa là `dp[s]`.

$$dp[p] = \max\Big(dp[p-1],\ \max_{(s,\,p,\,tip)} \big(dp[s] + p - s + tip\big)\Big),\quad dp[1] = 0.$$

**Phiên bản theo cuốc xe (khi tọa độ lớn):** sắp các cuốc theo `end`. `f[i]` = tiền tối đa dùng `i` cuốc đầu (theo end). Cuốc thứ `i` có `start = s`: số cuốc có `end ≤ s` là `j = bisect_right(ends, s)` ⇒
$$f[i] = \max(f[i-1],\ f[j] + profit_i).$$

**Quy tắc "nếu… thì…":** **nếu** hai cuốc có `a.end == b.start` **thì** chúng vẫn ghép được ⇒ dùng `bisect_right` (đếm `≤`), không phải `bisect_left`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP theo điểm trên đường `O(n + m)`

**Giải thích từng đoạn:**
1. Gom cuốc theo điểm kết thúc: `by_end[e] = [(s, profit), ...]`.
2. `dp = [0] * (n + 1)`; với `p = 2..n`: `dp[p] = dp[p−1]`, rồi thử mọi cuốc kết thúc tại `p`.
3. Trả `dp[n]`.

In [7]:
def earn_points(n, rides):
    by_end = defaultdict(list)
    for s, e, t in rides:
        by_end[e].append((s, e - s + t))
    dp = [0] * (n + 1)
    for p in range(2, n + 1):
        dp[p] = dp[p - 1]
        for s, pr in by_end[p]:
            if dp[s] + pr > dp[p]:
                dp[p] = dp[s] + pr
    return dp[n]

for n, rides, exp in examples:
    got = earn_points(n, rides); print(got); assert got == exp

7
20


In [8]:
class SolutionPoints:
    def maxTaxiEarnings(self, n: int, rides: List[List[int]]) -> int:
        by_end = defaultdict(list)
        for s, e, t in rides:
            by_end[e].append((s, e - s + t))
        dp = [0] * (n + 1)
        for p in range(2, n + 1):
            dp[p] = dp[p - 1]
            for s, pr in by_end[p]:
                dp[p] = max(dp[p], dp[s] + pr)
        return dp[n]

print([SolutionPoints().maxTaxiEarnings(n, r) for n, r, _ in examples])

[7, 20]


**Độ phức tạp Cách 2:** mỗi điểm `O(1)` + mỗi cuốc được xét đúng một lần ⇒ `O(n + m)` thời gian, `O(n + m)` bộ nhớ.

### 7.2 Cách 3 — Sắp theo `end` + tìm kiếm nhị phân `O(m log m)`

Không phụ thuộc `n` (hữu ích nếu tọa độ tới `10⁹`).

**Giải thích từng đoạn:**
1. `rides.sort(key=end)`, `ends = [e ...]`.
2. `f[0] = 0`; với cuốc thứ `i` (1-indexed) `(s, e, t)`: `j = bisect_right(ends, s, 0, i − 1)` (chỉ tìm trong các cuốc trước nó) — số cuốc kết thúc `≤ s`.
3. `f[i] = max(f[i−1], f[j] + e − s + t)`.

In [9]:
def earn_bisect(n, rides):
    rs = sorted(rides, key=lambda r: r[1])
    ends = [r[1] for r in rs]
    m = len(rs)
    f = [0] * (m + 1)
    for i in range(1, m + 1):
        s, e, t = rs[i - 1]
        j = bisect_right(ends, s, 0, i - 1)      # số cuốc (trong i-1 cuốc trước) có end <= s
        f[i] = max(f[i - 1], f[j] + e - s + t)
    return f[m]

for n, rides, exp in examples:
    got = earn_bisect(n, rides); print(got); assert got == exp

7
20


In [10]:
class SolutionBisect:
    def maxTaxiEarnings(self, n: int, rides: List[List[int]]) -> int:
        rides = sorted(rides, key=lambda r: r[1])
        ends = [r[1] for r in rides]
        f = [0] * (len(rides) + 1)
        for i, (s, e, t) in enumerate(rides, 1):
            j = bisect_right(ends, s, 0, i - 1)
            f[i] = max(f[i - 1], f[j] + e - s + t)
        return f[-1]

print([SolutionBisect().maxTaxiEarnings(n, r) for n, r, _ in examples])

[7, 20]


**Vì sao giới hạn `hi = i − 1`?** Vì `s < e` nên mọi cuốc có `end ≤ s` đều có `end < e` và nằm trước cuốc `i` trong thứ tự sắp theo end — thật ra không giới hạn cũng ra cùng kết quả, nhưng giới hạn giúp rõ ràng "chỉ dựa vào các cuốc trước". **Độ phức tạp:** `O(m log m)` thời gian, `O(m)` bộ nhớ.

### 7.3 Cách 4 — Sắp theo `start`, DP từ phải sang trái (dạng "memo" khử đệ quy)

`g[i]` = tiền tối đa chỉ dùng các cuốc `i, i+1, ...` (đã sắp theo start). Chọn cuốc `i` ⇒ nhảy tới cuốc đầu tiên có `start ≥ end_i` (`bisect_left(starts, end_i)`). Viết vòng lặp ngược thay cho đệ quy (tránh vượt giới hạn đệ quy khi `m = 3·10⁴`).

In [11]:
def earn_by_start(n, rides):
    rs = sorted(rides)
    starts = [r[0] for r in rs]
    m = len(rs)
    g = [0] * (m + 1)
    for i in range(m - 1, -1, -1):
        s, e, t = rs[i]
        nxt = bisect_left(starts, e, i + 1)      # cuốc đầu tiên có start >= e
        g[i] = max(g[i + 1], e - s + t + g[nxt])
    return g[0]

for n, rides, exp in examples:
    got = earn_by_start(n, rides); print(got); assert got == exp

7
20


**Độ phức tạp Cách 4:** `O(m log m)`. Đây là bản "bottom-up" của lời giải đệ quy `dfs(i) = max(dfs(i+1), profit + dfs(next(i)))`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2. Sắp theo end: `(1,6,1) (3,10,2) (10,12,3) (11,12,2) (12,15,2) (13,18,1)`, `ends = [6,10,12,12,15,18]`.

| i | cuốc | profit | j = #end ≤ s | f[j] + profit | f[i−1] | f[i] |
|---|---|---|---|---|---|---|
| 1 | (1,6,1) | 6 | 0 | 6 | 0 | 6 |
| 2 | (3,10,2) | 9 | 0 | 9 | 6 | 9 |
| 3 | (10,12,3) | 5 | 2 | 14 | 9 | 14 |
| 4 | (11,12,2) | 3 | 2 | 12 | 14 | 14 |
| 5 | (12,15,2) | 5 | 4 | 19 | 14 | 19 |
| 6 | (13,18,1) | 6 | 4 | 20 | 19 | **20** |

In [12]:
def trace_bisect(n, rides):
    rs = sorted(rides, key=lambda r: r[1])
    ends = [r[1] for r in rs]
    f = [0] * (len(rs) + 1)
    print("ends =", ends)
    for i, (s, e, t) in enumerate(rs, 1):
        j = bisect_right(ends, s, 0, i - 1)
        take = f[j] + e - s + t
        f[i] = max(f[i - 1], take)
        print(f"i={i} cuốc=({s},{e},{t}) profit={e - s + t:2d} j={j} f[j]+profit={take:2d} "
              f"f[i-1]={f[i - 1]:2d} -> f[{i}]={f[i]} ({'chọn' if take >= f[i - 1] else 'bỏ'})")
    return f[-1]

print("Đáp án:", trace_bisect(*examples[1][:2]))

ends = [6, 10, 12, 12, 15, 18]
i=1 cuốc=(1,6,1) profit= 6 j=0 f[j]+profit= 6 f[i-1]= 0 -> f[1]=6 (chọn)
i=2 cuốc=(3,10,2) profit= 9 j=0 f[j]+profit= 9 f[i-1]= 6 -> f[2]=9 (chọn)
i=3 cuốc=(10,12,3) profit= 5 j=2 f[j]+profit=14 f[i-1]= 9 -> f[3]=14 (chọn)
i=4 cuốc=(11,12,2) profit= 3 j=2 f[j]+profit=12 f[i-1]=14 -> f[4]=14 (bỏ)
i=5 cuốc=(12,15,2) profit= 5 j=4 f[j]+profit=19 f[i-1]=14 -> f[5]=19 (chọn)
i=6 cuốc=(13,18,1) profit= 6 j=4 f[j]+profit=20 f[i-1]=19 -> f[6]=20 (chọn)
Đáp án: 20


In [13]:
def trace_points(n, rides):
    by_end = defaultdict(list)
    for s, e, t in rides:
        by_end[e].append((s, e - s + t))
    dp = [0] * (n + 1)
    for p in range(2, n + 1):
        dp[p] = dp[p - 1]
        for s, pr in by_end[p]:
            cand = dp[s] + pr
            print(f"p={p:2d}: cuốc từ {s} (profit {pr}) -> dp[{s}]+{pr} = {cand}", "(cải thiện)" if cand > dp[p] else "")
            dp[p] = max(dp[p], cand)
    print("dp =", dp[1:])
    return dp[n]

print("Đáp án:", trace_points(*examples[1][:2]))

p= 6: cuốc từ 1 (profit 6) -> dp[1]+6 = 6 (cải thiện)
p=10: cuốc từ 3 (profit 9) -> dp[3]+9 = 9 (cải thiện)
p=12: cuốc từ 10 (profit 5) -> dp[10]+5 = 14 (cải thiện)
p=12: cuốc từ 11 (profit 3) -> dp[11]+3 = 12 
p=15: cuốc từ 12 (profit 5) -> dp[12]+5 = 19 (cải thiện)
p=18: cuốc từ 13 (profit 6) -> dp[13]+6 = 20 (cải thiện)
dp = [0, 0, 0, 0, 0, 6, 6, 6, 6, 9, 9, 14, 14, 14, 19, 19, 19, 20, 20, 20]
Đáp án: 20


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxTaxiEarnings(self, n: int, rides: List[List[int]]) -> int:
        by_end = defaultdict(list)                 # điểm kết thúc -> [(start, lợi nhuận)]
        for s, e, t in rides:
            by_end[e].append((s, e - s + t))
        dp = [0] * (n + 1)                         # dp[p] = tiền tối đa khi đã tới điểm p
        for p in range(2, n + 1):
            best = dp[p - 1]                       # không có cuốc nào kết thúc tại p
            for s, pr in by_end[p]:
                if dp[s] + pr > best:              # rảnh ở s, chở khách tới p
                    best = dp[s] + pr
            dp[p] = best
        return dp[n]

for n, rides, exp in examples:
    assert Solution().maxTaxiEarnings(n, rides) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Định nghĩa chính xác: `OPT(p)` = tổng lợi nhuận lớn nhất của một tập cuốc **đôi một tương thích** mà mọi cuốc đều có `end ≤ p`. Ta chứng minh `dp[p] = OPT(p)` bằng quy nạp theo `p`.

- `p = 1`: không có cuốc nào có `end ≤ 1` (vì `start ≥ 1`, `end > start`) ⇒ `0`.
- Bước quy nạp: xét tập tối ưu `S` cho `OPT(p)`.
  - Nếu không cuốc nào trong `S` có `end = p` ⇒ `S` hợp lệ cho `p − 1` ⇒ `OPT(p) ≤ OPT(p−1) = dp[p−1]`.
  - Nếu có cuốc `r = (s, p, t) ∈ S`: chỉ có thể có **một** cuốc như vậy (hai cuốc cùng end chồng nhau vì đều có độ dài > 0). Mọi cuốc khác trong `S` tương thích với `r` và kết thúc `≤ p`, nên chúng đi **trước** `r` ⇒ `end ≤ s` ⇒ tổng của chúng `≤ OPT(s) = dp[s]` ⇒ `OPT(p) ≤ dp[s] + profit(r)`.
  - Ngược lại, mỗi giá trị trong công thức đều đạt được bởi một tập hợp lệ ⇒ `dp[p] ≤ OPT(p)`.

Cách 3 cùng lập luận, chỉ thay "điểm `p`" bằng "chỉ số cuốc theo end"; `bisect_right` đếm đúng các cuốc có `end ≤ s`, và nhờ sắp theo end, các cuốc đó là một **tiền tố**.

## 11. Phân tích Time / Space

Gọi `n` = số điểm, `m = len(rides)`.

| Cách | Time | Space | Phụ thuộc `n`? |
|---|---|---|---|
| 1. Brute force | `O(2^m · m log m)` | `O(m)` | không |
| 2. DP theo điểm | `O(n + m)` | `O(n + m)` | **có** |
| 3. Sort end + bisect | `O(m log m)` | `O(m)` | không |
| 4. Sort start + bisect | `O(m log m)` | `O(m)` | không |

Nếu `n` lên tới `10⁹` (biến thể), chỉ Cách 3/4 còn dùng được (hoặc nén tọa độ cho Cách 2).

In [15]:
rng = random.Random(2008)
n = 10**5
big = []
for _ in range(3 * 10**4):
    s = rng.randint(1, n - 1); e = rng.randint(s + 1, min(n, s + 2000)); big.append([s, e, rng.randint(1, 10**5)])
for name, fn in [("points", earn_points), ("bisect", earn_bisect), ("by_start", earn_by_start)]:
    t = time.perf_counter(); r = fn(n, big)
    print(f"{name:8s}: {r} ({time.perf_counter() - t:.3f}s)")

points  : 61802683 (0.038s)
bisect  : 61802683 (0.017s)
by_start: 61802683 (0.022s)


## 12. Bộ test

Ví dụ đề; biên (một cuốc, các cuốc nối đuôi tại cùng điểm, các cuốc lồng nhau, nhiều cuốc trùng hệt); và đối chiếu brute force trên dữ liệu ngẫu nhiên (`m ≤ 12`, `n ≤ 15`, seed cố định).

In [16]:
fns = {
    "brute": earn_brute, "points": earn_points, "bisect": earn_bisect, "by_start": earn_by_start,
    "Solution": lambda n, r: Solution().maxTaxiEarnings(n, r),
    "SolutionBisect": lambda n, r: SolutionBisect().maxTaxiEarnings(n, r),
    "SolutionPoints": lambda n, r: SolutionPoints().maxTaxiEarnings(n, r),
}
edge = [
    (2, [[1, 2, 1]], 2),                                   # một cuốc
    (4, [[1, 2, 1], [2, 3, 1], [3, 4, 1]], 6),             # nối đuôi tại cùng điểm
    (10, [[1, 10, 1], [2, 3, 1], [4, 5, 1]], 10),          # cuốc dài bao trùm thắng
    (10, [[1, 10, 1], [2, 3, 9], [4, 5, 9]], 20),          # hai cuốc ngắn thắng
    (5, [[1, 5, 3]] * 4, 7),                               # trùng hệt: chỉ chọn một
]
for n, rides, exp in [(n, r, e) for n, r, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(n, [list(x) for x in rides]) == exp, (name, n, rides)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [17]:
rng = random.Random(42)
for _ in range(300):
    n = rng.randint(2, 15)
    rides = []
    for _ in range(rng.randint(1, 12)):
        s = rng.randint(1, n - 1); e = rng.randint(s + 1, n)
        rides.append([s, e, rng.randint(1, 6)])
    want = earn_brute(n, rides)
    for name, fn in fns.items():
        assert fn(n, [list(x) for x in rides]) == want, (name, n, rides)
print("300 test ngẫu nhiên: mọi cách khớp brute force")

300 test ngẫu nhiên: mọi cách khớp brute force


In [18]:
rng = random.Random(5)
for _ in range(20):
    n = rng.randint(2, 3000)
    rides = []
    for _ in range(rng.randint(1, 2000)):
        s = rng.randint(1, n - 1); e = rng.randint(s + 1, n); rides.append([s, e, rng.randint(1, 10**5)])
    assert earn_points(n, rides) == earn_bisect(n, rides) == earn_by_start(n, rides)
print("20 test cỡ vừa: ba cách DP khớp nhau")

20 test cỡ vừa: ba cách DP khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` thay vì `≤` (không cho nối đuôi tại cùng điểm)

Dùng `bisect_left(ends, s)` đếm số cuốc `end < s`, bỏ mất trường hợp `end == s`. Ví dụ 2 cần nối `(3,10)` với `(10,12)`.

In [19]:
def wrong_strict(n, rides):
    rs = sorted(rides, key=lambda r: r[1]); ends = [r[1] for r in rs]
    f = [0] * (len(rs) + 1)
    for i, (s, e, t) in enumerate(rs, 1):
        j = bisect_left(ends, s, 0, i - 1)          # SAI: chỉ đếm end < s
        f[i] = max(f[i - 1], f[j] + e - s + t)
    return f[-1]

n, rides, exp = examples[1]
print("sai:", wrong_strict(n, rides), "| đúng:", exp)

sai: 18 | đúng: 20


In [20]:
print("sửa (bisect_right):", earn_bisect(n, rides))

sửa (bisect_right): 20


### Lỗi 2 — Tham lam theo "kết thúc sớm nhất" (như bài lập lịch không trọng số)

Tham lam kết thúc sớm nhất tối đa **số cuốc**, không phải **tiền**.

In [21]:
def wrong_greedy_end(n, rides):
    cur, total = 1, 0
    for s, e, t in sorted(rides, key=lambda r: r[1]):
        if s >= cur:
            total += e - s + t; cur = e
    return total

r = [[1, 3, 1], [2, 10, 5]]
print("sai:", wrong_greedy_end(10, r), "| đúng:", Solution().maxTaxiEarnings(10, r))

sai: 3 | đúng: 13


In [22]:
print("sửa (DP):", earn_bisect(10, [[1, 3, 1], [2, 10, 5]]))

sửa (DP): 13


### Lỗi 3 — Tham lam theo "lợi nhuận lớn nhất trước"

Chọn cuốc có lợi nhuận lớn nhất rồi loại các cuốc chồng với nó — có thể bỏ lỡ hai cuốc nhỏ cộng lại lớn hơn.

In [23]:
def wrong_greedy_profit(n, rides):
    chosen = []
    for r in sorted(rides, key=profit, reverse=True):
        if all(r[1] <= c[0] or c[1] <= r[0] for c in chosen):
            chosen.append(r)
    return sum(profit(c) for c in chosen)

r = [[1, 10, 1], [1, 5, 4], [5, 10, 4]]
print("sai:", wrong_greedy_profit(10, r), "| đúng:", Solution().maxTaxiEarnings(10, r))

sai: 10 | đúng: 17


In [24]:
print("sửa (DP):", earn_points(10, [[1, 10, 1], [1, 5, 4], [5, 10, 4]]))

sửa (DP): 17


### Lỗi 4 — Quên "kéo" giá trị: `dp[p] = dp[p−1]`

Nếu chỉ cập nhật `dp[p]` khi có cuốc kết thúc tại `p`, các điểm không có cuốc nào sẽ có `dp = 0`, và cuốc sau bắt đầu ở đó mất hết tiền trước.

In [25]:
def wrong_no_carry(n, rides):
    by_end = defaultdict(list)
    for s, e, t in rides: by_end[e].append((s, e - s + t))
    dp = [0] * (n + 1)
    for p in range(2, n + 1):
        for s, pr in by_end[p]:                    # SAI: thiếu dp[p] = dp[p-1]
            dp[p] = max(dp[p], dp[s] + pr)
    return max(dp)

n, rides, exp = examples[1]
print("sai:", wrong_no_carry(n, rides), "| đúng:", exp)

sai: 19 | đúng: 20


In [26]:
print("sửa:", earn_points(n, rides))

sửa: 20


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chọn các khoảng không chồng nhau, tối đa tổng trọng số"** ⇒ weighted interval scheduling ⇒ DP + sắp xếp + `bisect`.
- Tọa độ nhỏ (`≤ 10⁵`) ⇒ DP theo **điểm** đơn giản, `O(n + m)`. Tọa độ lớn ⇒ DP theo **khoảng đã sắp**.
- Đọc kỹ điều kiện ở **đầu mút**: được chạm (`≤`) hay không (`<`) ⇒ `bisect_right` hay `bisect_left`.
- Đệ quy có memo với `m = 3·10⁴` dễ vượt giới hạn đệ quy Python ⇒ viết vòng lặp ngược.
- Tham lam chỉ đúng cho bài **không trọng số** (đếm số khoảng tối đa).

## 15. Biến thể của bài

1. **LeetCode 1235 — Maximum Profit in Job Scheduling:** y hệt, lợi nhuận cho sẵn, tọa độ tới `10⁹` ⇒ Cách 3.
2. **LeetCode 1751 — Maximum Number of Events That Can Be Attended II:** tối đa `k` sự kiện, đầu mút **không** được chạm ⇒ DP 2 chiều `f[i][c]` + `bisect_left`/`bisect_right` phù hợp.
3. **LeetCode 435 — Non-overlapping Intervals:** không trọng số ⇒ tham lam theo end.
4. **Trả về danh sách cuốc đã chọn:** truy vết ngược bảng `f`.
5. **Có hai taxi** (chở tối đa 2 khách một lúc): bài khó hơn nhiều (luồng chi phí tối thiểu).

**Minh họa biến thể 1 (tọa độ lớn, kiểu LeetCode 1235):** tọa độ tới `10⁹` thì mảng `dp[0..n]` quá lớn. Hai lối thoát: (a) dùng thẳng Cách 3 (không phụ thuộc `n`); (b) **nén tọa độ** — chỉ giữ các điểm là đầu mút của cuốc nào đó, sắp xếp, đánh số lại `0..K−1`, rồi chạy DP theo điểm trên các chỉ số mới. Ô dưới cài (b) và kiểm với (a) trên dữ liệu tự dựng.

In [27]:
def earn_compressed(rides):
    pts = sorted({x for s, e, _ in rides for x in (s, e)})
    idx = {x: i for i, x in enumerate(pts)}          # tọa độ -> chỉ số nén
    by_end = defaultdict(list)
    for s, e, t in rides:
        by_end[idx[e]].append((idx[s], e - s + t))
    dp = [0] * len(pts)
    for p in range(1, len(pts)):
        dp[p] = dp[p - 1]
        for sp, pr in by_end[p]:
            dp[p] = max(dp[p], dp[sp] + pr)
    return dp[-1]

rng = random.Random(1235)
for _ in range(100):
    rides = []
    for _ in range(rng.randint(1, 300)):
        s = rng.randint(1, 10**9 - 1); e = rng.randint(s + 1, min(10**9, s + 10**8)); rides.append([s, e, rng.randint(1, 100)])
    assert earn_compressed(rides) == earn_bisect(10**9, rides)
print("Nén tọa độ khớp Cách 3 trên 100 bộ tọa độ tới 10^9; ví dụ 2 ->", earn_compressed(examples[1][1]))

Nén tọa độ khớp Cách 3 trên 100 bộ tọa độ tới 10^9; ví dụ 2 -> 20


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi tập con | `O(2^m·m log m)` | `O(m)` | kiểm tra, `m ≤ 14` |
| 2. DP theo điểm | `dp[p] = max(dp[p−1], dp[s]+profit)` | `O(n + m)` | `O(n + m)` | **nộp bài** khi `n` nhỏ |
| 3. Sort end + `bisect_right` | `f[i] = max(f[i−1], f[j]+profit)` | `O(m log m)` | `O(m)` | tọa độ lớn, tổng quát |
| 4. Sort start + `bisect_left` | DP từ phải sang | `O(m log m)` | `O(m)` | khi nghĩ theo đệ quy "từ i trở đi" |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `n = 10, rides = [[1,10,1],[1,5,1],[5,10,1]]`.
2. **Tính tay:** bảng `dp[1..6]` (Cách 2) cho `n = 6, rides = [[1,3,2],[2,5,1],[3,6,1],[1,6,0]]`.
3. **Điền TODO:** `chosen_rides(n, rides)` trả về **danh sách cuốc** của một lời giải tối ưu (truy vết Cách 3).
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `max_with_k(n, rides, k)` — chở **tối đa `k`** khách. Kiểm bằng brute force.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def chosen_rides(n, rides):
    # TODO: tính f như Cách 3, rồi đi ngược i = m..1: nếu f[i] == f[i-1] thì bỏ, ngược lại chọn và nhảy tới j
    pass

In [29]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_bisect_solution(n, rides):
    # TODO
    pass

In [30]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_with_k(n, rides, k):
    # TODO: f[c][i] = tốt nhất dùng i cuốc đầu (theo end) với tối đa c khách
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Cuốc dài: `9 + 1 = 10`. (2) Hai cuốc ngắn nối đuôi tại 5: `(4+1) + (5+1)`. (3) So sánh `10` và `11`.

**Bài 2.** (1) Lợi nhuận: `4, 4, 4, 5`. (2) `dp[3] = 4`, `dp[5] = max(4, dp[2]+4 = 4) = 4`. (3) `dp[6] = max(4, dp[3]+4 = 8, dp[1]+5 = 5) = 8`.

**Bài 3.** (1) Lưu `j` của từng `i`. (2) Đi ngược từ `i = m`. (3) Đảo ngược danh sách khi xong.

**Bài 4.** (1) Sắp theo end. (2) `bisect_right(ends, s, 0, i-1)`. (3) `f[i] = max(f[i-1], f[j] + profit)`.

**Bài 5.** (1) Thêm chiều `c = 0..k`. (2) `f[c][i] = max(f[c][i−1], f[c−1][j] + profit_i)`. (3) Đáp án `f[k][m]`.

## 19. Đáp án tham khảo

In [31]:
# Bài 1, 2
assert Solution().maxTaxiEarnings(10, [[1, 10, 1], [1, 5, 1], [5, 10, 1]]) == 11
assert Solution().maxTaxiEarnings(6, [[1, 3, 2], [2, 5, 1], [3, 6, 1], [1, 6, 0]]) == 8
print("Bài 1: 11 | Bài 2: 8")
trace_points(6, [[1, 3, 2], [2, 5, 1], [3, 6, 1], [1, 6, 0]])

Bài 1: 11 | Bài 2: 8
p= 3: cuốc từ 1 (profit 4) -> dp[1]+4 = 4 (cải thiện)
p= 5: cuốc từ 2 (profit 4) -> dp[2]+4 = 4 
p= 6: cuốc từ 3 (profit 4) -> dp[3]+4 = 8 (cải thiện)
p= 6: cuốc từ 1 (profit 5) -> dp[1]+5 = 5 
dp = [0, 0, 4, 4, 4, 8]


8

In [32]:
# Bài 3, 4
def chosen_rides_ans(n, rides):
    rs = sorted(rides, key=lambda r: r[1]); ends = [r[1] for r in rs]
    m = len(rs); f = [0] * (m + 1); jump = [0] * (m + 1)
    for i, (s, e, t) in enumerate(rs, 1):
        jump[i] = bisect_right(ends, s, 0, i - 1)
        f[i] = max(f[i - 1], f[jump[i]] + e - s + t)
    res, i = [], m
    while i > 0:
        if f[i] == f[i - 1]:
            i -= 1
        else:
            res.append(rs[i - 1]); i = jump[i]
    return res[::-1]

my_bisect_solution_ans = earn_bisect

rng = random.Random(33)
for _ in range(200):
    n = rng.randint(2, 15)
    rides = [[s, rng.randint(s + 1, n), rng.randint(1, 5)] for s in (rng.randint(1, n - 1) for _ in range(rng.randint(1, 10)))]
    ch = chosen_rides_ans(n, rides)
    assert all(ch[k][1] <= ch[k + 1][0] for k in range(len(ch) - 1))
    assert sum(profit(r) for r in ch) == earn_brute(n, rides) == my_bisect_solution_ans(n, rides)
print("Bài 3, 4: OK; ví dụ 2 chọn:", chosen_rides_ans(*examples[1][:2]))

Bài 3, 4: OK; ví dụ 2 chọn: [[3, 10, 2], [10, 12, 3], [13, 18, 1]]


In [33]:
# Bài 5
def max_with_k_ans(n, rides, k):
    rs = sorted(rides, key=lambda r: r[1]); ends = [r[1] for r in rs]; m = len(rs)
    f = [[0] * (m + 1) for _ in range(k + 1)]
    for c in range(1, k + 1):
        for i, (s, e, t) in enumerate(rs, 1):
            j = bisect_right(ends, s, 0, i - 1)
            f[c][i] = max(f[c][i - 1], f[c - 1][j] + e - s + t)
    return f[k][m]

def brute_k(n, rides, k):
    best = 0
    for size in range(0, min(k, len(rides)) + 1):
        for comb in combinations(rides, size):
            ch = sorted(comb)
            if all(ch[a][1] <= ch[a + 1][0] for a in range(len(ch) - 1)):
                best = max(best, sum(profit(r) for r in ch))
    return best

rng = random.Random(44)
for _ in range(200):
    n = rng.randint(2, 12); k = rng.randint(1, 4)
    rides = [[s, rng.randint(s + 1, n), rng.randint(1, 5)] for s in (rng.randint(1, n - 1) for _ in range(rng.randint(1, 9)))]
    assert max_with_k_ans(n, rides, k) == brute_k(n, rides, k)
print("Bài 5: OK; ví dụ 2 với k=1,2,3:", [max_with_k_ans(*examples[1][:2], k) for k in (1, 2, 3)])

Bài 5: OK; ví dụ 2 với k=1,2,3: [9, 15, 20]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template weighted interval scheduling:**

```python
items.sort(key=lambda x: x.end)
ends = [x.end for x in items]
f = [0] * (m + 1)
for i, (s, e, w) in enumerate(items, 1):
    j = bisect_right(ends, s, 0, i - 1)     # end <= s  (được chạm đầu mút)
    f[i] = max(f[i - 1], f[j] + w)
return f[m]
```

In [34]:
def template(items):
    items = sorted(items, key=lambda x: x[1]); ends = [x[1] for x in items]
    f = [0] * (len(items) + 1)
    for i, (s, e, w) in enumerate(items, 1):
        f[i] = max(f[i - 1], f[bisect_right(ends, s, 0, i - 1)] + w)
    return f[-1]
print([template([(s, e, e - s + t) for s, e, t in r]) for _, r, _ in examples])

[7, 20]


**Checklist:**

- [ ] Lợi nhuận = `end − start + tip` (đừng quên phần quãng đường).
- [ ] Được nối đuôi tại cùng điểm ⇒ `end ≤ start` ⇒ `bisect_right`.
- [ ] DP theo điểm: luôn `dp[p] = dp[p−1]` trước khi xét các cuốc kết thúc tại `p`.
- [ ] Không dùng tham lam (end sớm nhất / lợi nhuận lớn nhất).
- [ ] Không dùng đệ quy sâu (`m = 3·10⁴`) — viết vòng lặp.
- [ ] Đã thử: một cuốc, cuốc trùng nhau, cuốc lồng nhau, nối đuôi.